# Bronze
Dado armazenado da mesma maneira que foi extraído

In [52]:
import pandas as pd

bronze = pd.read_excel("data/pedidos_metalurgica_mezzalira.xlsx")
bronze

,Pedido,Data,Cliente,Cidade,Produto,Valor (R$)
0,1001.0,2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""",3260
1,1002.0,2026-01-02 00:00:00,Grades Caruaru,Caruaru,"Barra chata 1""",3570
2,1003.0,2026-01-06 00:00:00,Serralheria Capibaribe,recife,Telha trapezoidal,1940
3,1004.0,2026-01-07 00:00:00,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,4910
4,1005.0,2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,4500
5,1006.0,2026-01-27,Irrigação Vale Forte,Petrolina,"Cantoneira 1/2""",1350
6,1007.0,01-fev-2026,Grades Caruaru,Caruaru,Perfil U enrijecido,5360
7,1008.0,2026-02-04 00:00:00,Estaleiro Porto Novo,REC,"Cantoneira 1/2""",3900
8,1009.0,2026-02-06 00:00:00,Serralheria Capibaribe,Recife,"Barra chata 1""",4040
9,1010.0,2026-02-07 00:00:00,Calhas Olinda,Olinda,"Tubo galvanizado 2""",5700


# Logger


In [53]:
logs = []
def logger(o_que_encontrei, o_que_fiz, linhas_afetadas, justificativa=""):
    logs.append({
        "O que encontrei": o_que_encontrei,
        "O que fiz": o_que_fiz,
        "Linhas afetadas": int(linhas_afetadas),
        "Justificativa (se houve dúvida)": justificativa,
    })
    print(f"✔ {o_que_encontrei}: {linhas_afetadas} linha(s) afetada(s)")

# Prata
Correções e manipulações dos dados

In [54]:
prata = bronze.copy()

columnsNames = bronze.columns.tolist()
for column in columnsNames[1:]:
    print("\n"+column+"\n")
    for valor in prata[column].unique():
        print(valor)


Data

2026-01-01
2026-01-02 00:00:00
2026-01-06 00:00:00
2026-01-07 00:00:00
2026-01-16
2026-01-27
01-fev-2026
2026-02-04 00:00:00
2026-02-06 00:00:00
2026-02-07 00:00:00
2026-02-13 00:00:00
2026-02-15 00:00:00
2026-02-16 00:00:00
2026-02-17 00:00:00
nan
2026-02-19
2026-02-20 00:00:00
2026-02-23 00:00:00
2026-02-25 00:00:00
2026-02-26
2026-02-26 00:00:00
2026-02-27
2026-03-05
2026-03-05 00:00:00
2026-03-06 00:00:00
06-mar-2026
2026-03-07 00:00:00
2026-03-10 00:00:00
2026-03-12 00:00:00
12-mar-2026
2026-03-14
2026-03-15 00:00:00
15-mar-2026
2026-03-16 00:00:00
19-mar-2026
2026-03-20 00:00:00
20-mar-2026
2026-03-21
2026-03-22 00:00:00
2026-03-25 00:00:00
2026-03-26 00:00:00
2026-03-27 00:00:00
2026-04-01 00:00:00
2026-04-05 00:00:00
2026-04-10 00:00:00
2026-04-13
2026-04-20 00:00:00
2026-04-23 00:00:00
2026-04-26 00:00:00

Cliente

Irrigação Vale Forte
Grades Caruaru
Serralheria Capibaribe
Agro Estruturas São Francisco
Estaleiro Porto Novo
Calhas Olinda
nan
Estruturas Piedade
Construtor

Remoção das linhas com valores "Nan"


In [55]:
antes = len(prata)
prata = prata.dropna()
logger(
    "Linhas totalmente vazias",
    "Removidas linhas sem nenhum campo preenchido",
    antes - len(prata),
    "Linhas parcialmente preenchidas foram preservadas para evitar excluir pedidos que ainda podem ser recuperados."
)

✔ Linhas totalmente vazias: 3 linha(s) afetada(s)


Remoção dos pedidos duplicados

In [56]:
antes = len(prata)
prata = prata.drop_duplicates(subset="Pedido", keep="first")
logger(
    "Pedidos com número duplicado",
    "Mantido o primeiro registro de cada número de pedido",
    antes - len(prata),
    "Sem um campo de atualização ou outro critério, foi preservada a primeira ocorrência."
)

✔ Pedidos com número duplicado: 4 linha(s) afetada(s)


Padronização dos nomes das cidades

In [57]:
correctNames = {
    "recife": "Recife",
    "REC": "Recife",
}

prata["Cidade"].value_counts()

antes = prata["Cidade"].copy()

prata["Cidade"] = prata["Cidade"].replace(correctNames)

linhasCorrigidas = (antes != prata["Cidade"]).sum()
logger(
    "Cidade com grafias diferentes (recife/REC)",
    "Padronizadas as grafias para Recife",
    linhasCorrigidas,
    "Assumiu-se que recife e REC representam o município de Recife."
)

prata["Cidade"].value_counts()

✔ Cidade com grafias diferentes (recife/REC): 10 linha(s) afetada(s)


Cidade
Recife                     16
Caruaru                    11
Olinda                      9
Jaboatão dos Guararapes     9
Petrolina                   8
Name: count, dtype: int64

Padronização dos valores

In [58]:
antes = prata["Valor (R$)"].copy()

valorLimpo = (
    prata["Valor (R$)"]
    .astype("string")
    .str.replace(r"R\$\s*", "", regex=True)
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .str.strip()
)

prata["Valor (R$)"] = pd.to_numeric(valorLimpo, errors="coerce")

linhasCorrigidas = (
    antes.notna() & (antes.astype("string") != prata["Valor (R$)"].astype("string"))
).sum()

logger(
    "Valores despadronizados",
    "Convertidos para números decimais",
    linhasCorrigidas,
    "Valores no padrão brasileiro foram interpretados com ponto como separador de milhar e vírgula como separador decimal."
)

✔ Valores despadronizados: 53 linha(s) afetada(s)


Padronização das datas


In [59]:
meses = {
    "jan": "01", "fev": "02", "mar": "03", "abr": "04",
    "mai": "05", "jun": "06", "jul": "07", "ago": "08",
    "set": "09", "out": "10", "nov": "11", "dez": "12",
}

antes = prata["Data"].copy()

datas = (
    prata["Data"]
    .astype("string")
    .str.strip()
    .str.lower()
)

for mes, numero in meses.items():
    datas = datas.str.replace(mes, numero, regex=False)

datas_convertidas = pd.Series(pd.NaT, index=datas.index, dtype="datetime64[ns]")
for formato in ["%Y-%m-%d %H:%M:%S", "%Y-%m-%d",
                "%d-%m-%Y", "%d/%m/%Y", "%d/%m/%y"]:
    datas_convertidas = datas_convertidas.fillna(
        pd.to_datetime(datas, format=formato, errors="coerce")
    )

prata["Data"] = datas_convertidas.dt.strftime("%d/%m/%y").astype("string")
linhasConvertidas = (antes.notna() & prata["Data"].notna() & (antes.astype("string") != prata["Data"])).sum()

linhas_invalidas = (antes.notna() & prata["Data"].isna()).sum()

logger(
    "Datas em formatos diferentes",
    f"Padronizadas para dd/mm/aa: {linhasConvertidas}; inválidas para revisão: {linhas_invalidas}",
    linhasConvertidas,
    "Abreviações de meses em português foram convertidas para números; datas não interpretadas precisam de revisão."
)

prata[["Data"]]

✔ Datas em formatos diferentes: 53 linha(s) afetada(s)


,Data
0,01/01/26
1,02/01/26
2,06/01/26
3,07/01/26
4,16/01/26
5,27/01/26
6,01/02/26
7,04/02/26
8,06/02/26
9,07/02/26


# Ouro


In [60]:
base_ouro = prata.copy()
base_ouro["Mês"] = pd.to_datetime(base_ouro["Data"], format="%d/%m/%y", errors="raise").dt.month.map({
        1: "Janeiro", 2: "Fevereiro", 3: "Março", 4: "Abril",
        5: "Maio", 6: "Junho", 7: "Julho", 8: "Agosto",
        9: "Setembro", 10: "Outubro", 11: "Novembro", 12: "Dezembro"
    })
ordem_meses = ["Janeiro", "Fevereiro", "Março", "Abril",
                   "Maio", "Junho", "Julho", "Agosto", "Setembro",
                   "Outubro", "Novembro", "Dezembro"]
ouro = base_ouro.pivot_table(
        index="Cidade", columns="Mês", values="Valor (R$)",
        aggfunc="sum", fill_value=0, margins=True, margins_name="Total"
    ).reindex(columns=ordem_meses + ["Total"], fill_value=0).reset_index()

# Exportação

In [61]:
colunas_log = [
    "O que encontrei",
    "O que fiz",
    "Linhas afetadas",
    "Justificativa (se houve dúvida)",
]
log_df = pd.DataFrame(logs, columns=colunas_log)

arquivo_saida = "data/pedidos_metalurgica_tratados.xlsx"
with pd.ExcelWriter(arquivo_saida, engine="openpyxl") as writer:
    bronze.to_excel(writer, sheet_name="Bronze", index=False)
    prata.to_excel(writer, sheet_name="Prata", index=False)
    ouro.to_excel(writer, sheet_name="Ouro", index=False)
    log_df.to_excel(writer, sheet_name="Log", index=False)
    
print(f"Arquivo salvo em: {arquivo_saida}")
log_df

Arquivo salvo em: data/pedidos_metalurgica_tratados.xlsx


,O que encontrei,O que fiz,Linhas afetadas,Justificativa (se houve dúvida)
0,Linhas totalmente vazias,Removidas linhas sem nenhum campo preenchido,3,Linhas parcialmente preenchidas foram preserva...
1,Pedidos com número duplicado,Mantido o primeiro registro de cada número de ...,4,"Sem um campo de atualização ou outro critério,..."
2,Cidade com grafias diferentes (recife/REC),Padronizadas as grafias para Recife,10,Assumiu-se que recife e REC representam o muni...
3,Valores despadronizados,Convertidos para números decimais,53,Valores no padrão brasileiro foram interpretad...
4,Datas em formatos diferentes,Padronizadas para dd/mm/aa: 53; inválidas para...,53,Abreviações de meses em português foram conver...


In [62]:
vendas_mensais = prata.copy()

vendas_mensais["Mês"] = pd.to_datetime(vendas_mensais["Data"], format="%d/%m/%y").dt.to_period("M")

faturamento_mensal = vendas_mensais.groupby(["Cidade", "Mês"])["Valor (R$)"].sum()

cidade_campea, mes_campeao = faturamento_mensal.idxmax()

valor_campeao = faturamento_mensal.loc[(cidade_campea, mes_campeao)]


nomes_meses = ["Janeiro", "Fevereiro", "Março", "Abril", "Maio", "Junho",
               "Julho", "Agosto", "Setembro", "Outubro", "Novembro", "Dezembro"]

def formatar_reais(valor):
    return (f"R$ {valor:,.2f}"
            .replace(",", "_")
            .replace(".", ",")
            .replace("_", "."))

print(f"A cidade que mais comprou em um único mês foi {cidade_campea}, em {nomes_meses[mes_campeao.month - 1]} de {mes_campeao.year}, com {formatar_reais(valor_campeao)}.")

A cidade que mais comprou em um único mês foi Recife, em Março de 2026, com R$ 29.730,00.
